# ラボ 1a: 出生（natality）データセットの探索

**学習目標**

1. BigQuery を使用して出生データセットを探索する
1. Vertex AI Workbench（Notebooks）を使用して探索的データをプロットする


## はじめに
このノートブックでは、赤ちゃんが生まれる前の体重を予測するモデルの開発とトレーニングを開始する前に、出生（natality）データセットを探索します。BigQuery を使用してデータを探索し、Vertex AI Workbench を使用して探索的データをプロットします。

## 必要なライブラリの読み込み

In [ ]:
from google.cloud import bigquery

## ソース データセット

データセットは [BigQuery](https://cloud.google.com/bigquery/) でホストされています。CDC の出生（Natality）データには 1969 年から 2008 年までの米国の出生に関する詳細が含まれており、一般公開されているデータセットであるため、Google Cloud アカウントを持つ誰でもアクセスできます。[こちら](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) をクリックしてデータセットにアクセスしてください。

出生データセットは約 1 億 3,800 万行、31 列と比較的規模が大きいですが、理解しやすい内容です。`weight_pounds`（ポンド単位の体重）がターゲットであり、モデルが予測するようにトレーニングする連続値です。

<h2> データの探索 </h2>

このデータは出生データ（米国における出生記録）です。目的は、妊娠や赤ちゃんの母親に関する多くの要因を踏まえて、赤ちゃんの体重を予測することです。後でデータをトレーニング用データセットと評価用データセットに分割します。その際、年月（year-month）のハッシュを使用します。これにより、同じ日に生まれた双子がデータの異なる分割に分かれてしまうのを防ぐことができます。まず、2000 年以降の出生データを使用した SQL クエリを作成します。

In [ ]:
query = """
SELECT
    weight_pounds,
    is_male,
    mother_age,
    plurality,
    gestation_weeks,
    FARM_FINGERPRINT(
        CONCAT(
            CAST(YEAR AS STRING),
            CAST(month AS STRING)
        )
    ) AS hashmonth
FROM
    publicdata.samples.natality
WHERE
    year > 2000
"""

ノートブック全体で使用できる BigQuery クライアントを作成しましょう。

In [ ]:
bq = bigquery.Client()

新しく作成したクライアントを使用して、BigQuery の呼び出し結果を Pandas DataFrame で確認してみましょう。

In [ ]:
df = bq.query(query + " LIMIT 100").to_dataframe()
df.head()

まず、出生データセット内のすべての有効な列名のセットを取得しましょう。これを行うには、データセットのテーブルの `INFORMATION_SCHEMA` にアクセスします。

In [ ]:
# テーブル スキーマ内のすべての列名を取得するクエリ
sql = """
SELECT
    column_name
FROM
    publicdata.samples.INFORMATION_SCHEMA.COLUMNS
WHERE
    table_name = "natality"
"""

# BigQuery クライアント経由でクエリを送信し、出力を DataFrame に格納
valid_columns_df = bq.query(sql).to_dataframe()

# DataFrame 内の列名をセットに変換
valid_columns_set = valid_columns_df["column_name"].tolist()

有効な列のセットを出力して、データセットで使用可能なすべての列を確認できます。もちろん、[BigQuery UI](https://console.cloud.google.com/bigquery?project=bigquery-public-data&p=publicdata&d=samples&t=natality&page=table) でテーブルを選択したときに [スキーマ] タブに移動してこの情報を確認することもできます。

In [ ]:
print(valid_columns_set)

各列の一意の値とその値のカウントを調べるクエリを作成しましょう。
これは、各データ値の十分なサンプルがあることを確認し、そのパラメータに予測価値があるという直感を検証するために重要です。

In [ ]:
def get_distinct_values(valid_columns_set, column_name):
    """BigQuery データ列の一意の値に関する統計情報を取得します。

    Args:
        valid_columns_set: set, the set of all possible valid column names in
            table.
        column_name: str、BigQuery 内の列名。
    Returns:
        一意の値、そのカウント、および平均値を含む DataFrame。
    """
    assert (
        column_name in valid_columns_set
    ), "{column_name} is not a valid column_name".format(
        column_name=column_name
    )

    sql = """
    SELECT
        {column_name},
        COUNT(1) AS num_babies,
        AVG(weight_pounds) AS avg_wt
    FROM
        publicdata.samples.natality
    WHERE
        year > 2000
    GROUP BY
        {column_name}
    """.format(
        column_name=column_name
    )

    return bq.query(sql).to_dataframe()

In [ ]:
def plot_distinct_values(valid_columns_set, column_name, logy=False):
    """BigQuery データ列の一意の値に関する統計情報をプロットします。

    Args:
        valid_columns_set: set, the set of all possible valid column names in
            table.
        column_name: str、BigQuery 内の列名。
        logy: bool、カウントを対数スケールでプロットするかどうか。
    """
    df = get_distinct_values(valid_columns_set, column_name)
    df = df.sort_values(column_name)
    df.plot(
        x=column_name, y="num_babies", logy=logy, kind="bar", figsize=(12, 5)
    )
    df.plot(x=column_name, y="avg_wt", kind="bar", figsize=(12, 5))

`is_male` について、`avg_wt`（平均体重）を線形スケール、`num_babies`（赤ちゃんの数）を対数スケールで表示する棒グラフを作成します。

In [ ]:
plot_distinct_values(valid_columns_set, column_name="is_male", logy=False)

`mother_age`（母親の年齢）について、`avg_wt` を線形スケール、`num_babies` を線形スケールで表示する棒グラフを作成します。

In [ ]:
plot_distinct_values(valid_columns_set, column_name="mother_age", logy=False)

`plurality`（出生数 / 単胎・多胎）について、`avg_wt` を線形スケール、`num_babies` を対数スケールで表示する棒グラフを作成します。

In [ ]:
plot_distinct_values(valid_columns_set, column_name="plurality", logy=True)

`gestation_weeks`（妊娠週数）について、`avg_wt` を線形スケール、`num_babies` を対数スケールで表示する棒グラフを作成します。

In [ ]:
plot_distinct_values(
    valid_columns_set, column_name="gestation_weeks", logy=True
)

これらすべての要因が赤ちゃんの体重に影響を与えているようです。男の子の赤ちゃんは平均して女の子の赤ちゃんよりも重いです。10 代や高齢の母親から生まれた赤ちゃんは体重が低くなる傾向があります。双子や三つ子などは単胎児よりも体重が低くなります。早産児（未熟児）は体重が軽くなり、未婚の母親から生まれた赤ちゃんも同様です。さらに、各入力値に対して十分なデータ（赤ちゃんの数）があるかどうかを確認することも重要です。そうしないと、十分なデータがない入力値に対するモデルの予測は信頼できない可能性があります。
<p>
次のノートブックでは、これらすべての要因を組み合わせて赤ちゃんの体重の予測を導き出す機械学習モデルを開発します。

## ラボのまとめ: 
このラボでは、BigQuery を使用してデータを探索し、Cloud AI Platform Notebooks（Vertex AI Workbench）を使用して探索的データをプロットしました。

Copyright 2021 Google LLC
Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at
    https://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.